# Whisper Lecture Transcription for Colab

금융계량(한국어 중심)과 회계원리(영어 중심 + 한국어 혼용) 강의 녹음을 Whisper로 1차 전사하는 노트북입니다.

권장 순서:
1. 설치 셀 실행
2. Google Drive 마운트
3. 설정값 입력
4. 함수 정의 셀 실행
5. 전사 실행
6. 필요하면 결과 zip 다운로드

In [ ]:
# 1) Install dependencies
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U openai-whisper

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 3) Configure paths and transcription mode

- `INPUT_PATH`: 오디오 파일 1개 또는 폴더
- `OUTPUT_DIR`: 결과 저장 폴더
- `MODE`:
  - `finance`: 한국어 중심 금융계량 강의
  - `accounting`: 영어 중심 + 한국어 혼용 회계원리 강의
  - `auto`: 자동 감지
- `MODEL_NAME`: `small`, `medium`, `large` 등

In [ ]:
# 3) User configuration
from pathlib import Path

# 예시:
# INPUT_PATH = Path('/content/drive/MyDrive/소리 녹음/금계')
# INPUT_PATH = Path('/content/drive/MyDrive/소리 녹음/회계원리/week05.m4a')

INPUT_PATH = Path('/content/drive/MyDrive/소리 녹음/금계')
OUTPUT_DIR = Path('/content/drive/MyDrive/소리 녹음/transcripts')

MODE = 'finance'       # finance | accounting | auto
MODEL_NAME = 'small'   # base | small | medium | large
TASK = 'transcribe'    # transcribe | translate
BEAM_SIZE = 5
TEMPERATURE = 0.0
APPEND_PRESET_PROMPT = ''

INPUT_PATH, OUTPUT_DIR

In [ ]:
# 4) Define helper functions
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import whisper

AUDIO_EXTENSIONS = {'.m4a', '.mp3', '.wav', '.mp4', '.aac', '.flac', '.ogg', '.webm'}


def resolve_audio_files(input_path: Path) -> list[Path]:
    if not input_path.exists():
        raise FileNotFoundError(f'입력 경로를 찾을 수 없습니다: {input_path}')

    if input_path.is_file():
        if input_path.suffix.lower() not in AUDIO_EXTENSIONS:
            raise ValueError(f'지원하지 않는 확장자입니다: {input_path.suffix}')
        return [input_path]

    files = sorted(
        path for path in input_path.rglob('*')
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )
    if not files:
        raise FileNotFoundError(f'오디오 파일을 찾지 못했습니다: {input_path}')
    return files


def choose_language(mode: str) -> str | None:
    if mode == 'finance':
        return 'ko'
    if mode == 'accounting':
        return None
    return None


def build_initial_prompt(mode: str, extra_prompt: str = '') -> str:
    prompts = {
        'finance': (
            '이 음성은 한국어 중심의 금융계량경제학 강의이다. '
            '계량경제학, 금융계량, 회귀분석, 시계열, 추정, 검정, 변수, 모형 같은 용어를 정확히 적는다. '
            '수식이나 영어 용어는 가능한 한 원어를 보존한다.'
        ),
        'accounting': (
            'This audio is an accounting lecture delivered mainly in English with Korean mixed in. '
            'Preserve accounting terms such as asset, liability, equity, revenue, expense, journal entry, '
            'debit, credit, balance sheet, income statement, cash flow, accrual, and adjustment. '
            '한국어 설명이 섞이면 자연스럽게 그대로 적는다.'
        ),
        'auto': (
            'This is a university lecture recording. '
            'Preserve specialized academic terms, English keywords, Korean explanations, and numbers accurately.'
        ),
    }
    return f"{prompts[mode]} {extra_prompt.strip()}".strip()


def format_timestamp(seconds: float) -> str:
    total_ms = int(round(seconds * 1000))
    hours, rem = divmod(total_ms, 3_600_000)
    minutes, rem = divmod(rem, 60_000)
    secs, ms = divmod(rem, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}'


def write_text_output(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.strip() + '\n', encoding='utf-8')


def write_segments_output(path: Path, segments: Iterable[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(list(segments), ensure_ascii=False, indent=2), encoding='utf-8')


def transcribe_one(model, audio_path: Path, output_dir: Path, mode: str) -> None:
    language = choose_language(mode)
    prompt = build_initial_prompt(mode, APPEND_PRESET_PROMPT)

    print(f'[START] {audio_path.name}')
    result = model.transcribe(
        str(audio_path),
        task=TASK,
        language=language,
        initial_prompt=prompt,
        beam_size=BEAM_SIZE,
        temperature=TEMPERATURE,
        verbose=False,
        fp16=False,
    )

    stem_dir = output_dir / audio_path.stem
    transcript_path = stem_dir / f'{audio_path.stem}.txt'
    segments_path = stem_dir / f'{audio_path.stem}.segments.json'

    write_text_output(transcript_path, result['text'])

    segments = []
    for segment in result.get('segments', []):
        segments.append({
            'id': segment.get('id'),
            'start': segment.get('start'),
            'end': segment.get('end'),
            'start_ts': format_timestamp(segment.get('start', 0.0)),
            'end_ts': format_timestamp(segment.get('end', 0.0)),
            'text': segment.get('text', '').strip(),
        })
    write_segments_output(segments_path, segments)
    print(f'[DONE] {audio_path.name} -> {transcript_path}')


In [ ]:
# 5) Load model
print(f'모델 로드 중: {MODEL_NAME}')
model = whisper.load_model(MODEL_NAME)
print('모델 로드 완료')

In [ ]:
# 6) Run transcription
audio_files = resolve_audio_files(INPUT_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for audio_path in audio_files:
    try:
        transcribe_one(model, audio_path, OUTPUT_DIR, MODE)
    except Exception as exc:
        print(f'[ERROR] {audio_path.name}: {exc}')

print(f'출력 폴더: {OUTPUT_DIR}')

## 7) Optional: zip outputs for download

Drive에 저장하면 보통 이 셀은 필요 없습니다. 바로 다운로드하고 싶을 때만 사용하세요.

In [ ]:
# 7) Optional: zip outputs and download
from google.colab import files
import shutil

zip_base = '/content/whisper_transcripts'
archive_path = shutil.make_archive(zip_base, 'zip', root_dir=str(OUTPUT_DIR))
files.download(archive_path)